In [ ]:
# # ============================================================
# # RUN THIS FIRST -- storage cleanup
# # Deletes stale/duplicate files from earlier failed runs of this
# # notebook so your Drive/Colab disk isn't clogged. Safe to run
# # any time; it only removes files this notebook itself creates.
# # ============================================================
# import os
# import shutil

# save_dir = '/content/drive/MyDrive/PEDA_Processed'
# os.makedirs(save_dir, exist_ok=True)

# print("=== Drive: /PEDA_Processed BEFORE cleanup ===")
# total_before = 0
# for f in sorted(os.listdir(save_dir)):
#     p = f'{save_dir}/{f}'
#     size = os.path.getsize(p) / (1024 * 1024)
#     total_before += size
#     print(f"  {f} ({size:.1f} MB)")
# print(f"Total: {total_before:.1f} MB\n")

# # Stale filenames from earlier (pre-fix) versions of this notebook --
# # safe to delete, nothing downstream reads these anymore.
# stale_files = [
#     'data_c2.pt', 'data_c20.pt', 'data_n14.pt',
#     'data_c2_list.pt', 'data_c20_list.pt', 'data_n14_list.pt',
#     'label_mean.npy', 'label_std.npy',
#     'data_c2_norm.pt', 'data_c20_norm.pt', 'data_n14_norm.pt',
# ]
# freed = 0
# for f in stale_files:
#     p = f'{save_dir}/{f}'
#     if os.path.exists(p):
#         freed += os.path.getsize(p) / (1024 * 1024)
#         os.remove(p)
#         print(f"Deleted: {f}")

# print(f"\nFreed {freed:.1f} MB from Drive.")

# # /content (Colab runtime disk) -- large intermediate extraction
# # folders (RUDY tar extracts, graph_N14/graph_N28) that don't need
# # to persist once *_raw.pt files exist on Drive.
# content_dirs = ['/content/labels_N14', '/content/labels_N28',
#                  '/content/graph_N14', '/content/graph_N28',
#                  '/content/RUDY_N14.tar.gz', '/content/RUDY_N28.tar.gz']
# for p in content_dirs:
#     if os.path.exists(p):
#         size = (sum(os.path.getsize(os.path.join(dp, f))
#                      for dp, _, fs in os.walk(p) for f in fs)
#                 if os.path.isdir(p) else os.path.getsize(p)) / (1024 * 1024)
#         if os.path.isdir(p):
#             shutil.rmtree(p)
#         else:
#             os.remove(p)
#         print(f"Deleted (runtime disk): {p} ({size:.1f} MB)")

# print("\n=== Drive: /PEDA_Processed AFTER cleanup ===")
# total_after = 0
# for f in sorted(os.listdir(save_dir)):
#     p = f'{save_dir}/{f}'
#     size = os.path.getsize(p) / (1024 * 1024)
#     total_after += size
#     print(f"  {f} ({size:.1f} MB)")
# print(f"Total: {total_after:.1f} MB")

# total, used, free = shutil.disk_usage('/content')
# print(f"\nColab runtime disk: {used/1e9:.1f}GB used / {free/1e9:.1f}GB free")
# print("\n\u2705 Cleanup done. Now run cells top to bottom.")


=== Drive: /PEDA_Processed BEFORE cleanup ===
Total: 0.0 MB


Freed 0.0 MB from Drive.

=== Drive: /PEDA_Processed AFTER cleanup ===
Total: 0.0 MB

Colab runtime disk: 50.5GB used / 70.4GB free

✅ Cleanup done. Now run cells top to bottom.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [2]:
# ============================================
# PEDA PROJECT - STARTUP CELL
# Run this at the beginning of every session!
# ============================================

from google.colab import drive
drive.mount('/content/drive')

!pip install torch-geometric -q

import torch
import numpy as np
from torch_geometric.data import Data

save_dir = '/content/drive/MyDrive/PEDA_Processed'

# C2 / C20 : small enough to load as full lists of Data objects
data_c2  = torch.load(f'{save_dir}/data_c2_raw.pt',  weights_only=False)
data_c20 = torch.load(f'{save_dir}/data_c20_raw.pt', weights_only=False)

# N14 : compact representation (fix -- avoids the RAM crash from
# loading 3441 duplicated full-size graphs at once). Build one
# Data object at a time with make_n14_data(i, ...) when you need it.
n14_base_x     = torch.load(f'{save_dir}/n14_base_x.pt')
n14_edge_index = torch.load(f'{save_dir}/n14_edge_index.pt')
n14_configs    = np.load(f'{save_dir}/n14_configs_norm.npy')
n14_labels     = np.load(f'{save_dir}/n14_labels_raw.npy')

def make_n14_data(i):
    num_nodes = n14_base_x.shape[0]
    cfg = torch.tensor(n14_configs[i], dtype=torch.float).unsqueeze(0).repeat(num_nodes, 1)
    x = torch.cat([n14_base_x, cfg], dim=1)
    y = torch.tensor([n14_labels[i]], dtype=torch.float)
    return Data(x=x, edge_index=n14_edge_index, y=y)

# Load feature scaler (fit on C2 only)
scaler_mean = np.load(f'{save_dir}/scaler_mean.npy')
scaler_std  = np.load(f'{save_dir}/scaler_std.npy')

# Load per-domain label stats
label_c2_mean  = float(np.load(f'{save_dir}/label_c2_mean.npy'))
label_c2_std   = float(np.load(f'{save_dir}/label_c2_std.npy'))
label_c20_mean = float(np.load(f'{save_dir}/label_c20_mean.npy'))
label_c20_std  = float(np.load(f'{save_dir}/label_c20_std.npy'))
label_n14_mean = float(np.load(f'{save_dir}/label_n14_mean.npy'))
label_n14_std  = float(np.load(f'{save_dir}/label_n14_std.npy'))

print("=" * 45)
print("  PEDA PROJECT \u2014 ALL DATA LOADED!")
print("=" * 45)
print(f"  C2  : nodes={data_c2[0].x.shape[0]}  configs={len(data_c2)}")
print(f"  C20 : nodes={data_c20[0].x.shape[0]}  configs={len(data_c20)}")
print(f"  N14 : nodes={n14_base_x.shape[0]}  configs={len(n14_labels)}  (built on demand via make_n14_data(i))")
print(f"\n  Feature scaler mean : {scaler_mean}")
print(f"  Feature scaler std  : {scaler_std}")
print(f"\n  Label C2  mean/std  : {label_c2_mean:.6f} / {label_c2_std:.6f}")
print(f"  Label C20 mean/std  : {label_c20_mean:.6f} / {label_c20_std:.6f}")
print(f"  Label N14 mean/std  : {label_n14_mean:.6f} / {label_n14_std:.6f}")
print(f"\n\u2705 Ready to continue!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00
  PEDA PROJECT — ALL DATA LOADED!
  C2  : nodes=42102  configs=264
  C20 : nodes=41405  configs=271
  N14 : nodes=34983  configs=3441  (built on demand via make_n14_data(i))

  Feature scaler mean : [ 3.9944184 43.580032 ]
  Feature scaler std  : [ 1.1122626 84.87292  ]

  Label C2  mean/std  : 0.001838 / 0.000381
  Label C20 mean/std  : 0.001998 / 0.000759
  Label N14 mean/std  : 0.004918 / 0.000874

✅ Ready to continue!


In [3]:
import tarfile
import os

# Extract N28 graph_information directly from Drive into Colab /content
# NO copying the whole dataset - just extracting what we need!

print("Extracting N28 graph_information...")

tar_path = '/content/drive/MyDrive/CircuitNet/CircuitNet-N28/graph_features/graph_information.tar.gz'
extract_to = '/content/graph_N28/'

os.makedirs(extract_to, exist_ok=True)

with tarfile.open(tar_path, 'r:gz') as tar:
    # Only extract zero-riscy files to save space!
    members = [m for m in tar.getmembers() if 'zero-riscy' in m.name]
    print(f"Found {len(members)} zero-riscy files inside tar")
    for m in members:
        print(f"  {m.name}")
    tar.extractall(path=extract_to, members=members)

print("✅ Done! Extracted only zero-riscy files")

Extracting N28 graph_information...
Found 54 zero-riscy files inside tar
  net_attr/zero-riscy-b-1-c5_net_attr.npy
  net_attr/zero-riscy-b-3-c5_net_attr.npy
  net_attr/zero-riscy-b-2-c20_net_attr.npy
  net_attr/zero-riscy-b-1-c20_net_attr.npy
  net_attr/zero-riscy-b-2-c2_net_attr.npy
  net_attr/zero-riscy-b-2-c5_net_attr.npy
  net_attr/zero-riscy-b-3-c20_net_attr.npy
  net_attr/zero-riscy-b-3-c2_net_attr.npy
  net_attr/zero-riscy-b-1-c2_net_attr.npy
  net_attr/zero-riscy-a-2-c5_net_attr.npy
  net_attr/zero-riscy-a-1-c5_net_attr.npy
  net_attr/zero-riscy-a-3-c20_net_attr.npy
  net_attr/zero-riscy-a-3-c5_net_attr.npy
  net_attr/zero-riscy-a-1-c20_net_attr.npy
  net_attr/zero-riscy-a-2-c20_net_attr.npy
  net_attr/zero-riscy-a-1-c2_net_attr.npy
  net_attr/zero-riscy-a-2-c2_net_attr.npy
  net_attr/zero-riscy-a-3-c2_net_attr.npy
  node_attr/zero-riscy-b-1-c5_node_attr.npy
  node_attr/zero-riscy-b-3-c5_node_attr.npy
  node_attr/zero-riscy-b-2-c20_node_attr.npy
  node_attr/zero-riscy-b-1-c20_n

/tmp/ipykernel_3970/1815141248.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to, members=members)


✅ Done! Extracted only zero-riscy files


In [4]:
import os
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print(f)

CircuitNet
Colab Notebooks
PEDA_Processed
Untitled5.ipynb
preprocessing_fixed (4).ipynb


In [5]:
import numpy as np

# Load ONLY the 2 variants we decided on
VARIANT = 'zero-riscy-a-1-c2'
base = '/content/graph_N28'

node_data = np.load(f'{base}/node_attr/{VARIANT}_node_attr.npy', allow_pickle=True)
pin_data  = np.load(f'{base}/pin_attr/{VARIANT}_pin_attr.npy',  allow_pickle=True)
net_data  = np.load(f'{base}/net_attr/{VARIANT}_net_attr.npy',  allow_pickle=True)

print(f"✅ Variant: {VARIANT}")
print(f"   node_data shape : {node_data.shape}")
print(f"   pin_data  shape : {pin_data.shape}")
print(f"   net_data  shape : {net_data.shape}")

# Peek at what's inside node_data
print(f"\n   node_data row 0 : {node_data[0, :5]}")
print(f"   node_data row 1 : {node_data[1, :5]}")

✅ Variant: zero-riscy-a-1-c2
   node_data shape : (2, 42102)
   pin_data  shape : (3, 168173)
   net_data  shape : (1, 42428)

   node_data row 0 : ['clk_rst_gen_i/PLL_i' 'clk_rst_gen_i/i_rst_gen_soc/s_rst_ff3_reg'
 'clk_rst_gen_i/i_rst_gen_soc/s_rst_ff2_reg'
 'clk_rst_gen_i/i_rst_gen_soc/s_rst_ff1_reg'
 'clk_rst_gen_i/i_rst_gen_soc/s_rst_ff0_reg']
   node_data row 1 : ['PLL' 'DFCNQ_x1_0' 'DFCNQ_x1_0' 'DFCNQ_x1_0' 'DFCNQ_x1_0']


In [6]:
import shutil
import os

src = '/content/drive/MyDrive/CircuitNet/CircuitNet-N28/routability_features/RUDY/RUDY.tar.gz'
dst = '/content/RUDY.tar.gz'

# Check if already copied
if os.path.exists(dst):
    print(f"Already exists! Size: {os.path.getsize(dst)/(1024*1024):.1f} MB")
else:
    print("Copying RUDY.tar.gz to Colab local storage...")
    print("This will take ~5 minutes, please wait...")
    shutil.copy2(src, dst)
    print(f"✅ Done! Size: {os.path.getsize(dst)/(1024*1024):.1f} MB")

Copying RUDY.tar.gz to Colab local storage...
This will take ~5 minutes, please wait...
✅ Done! Size: 2607.5 MB


In [7]:
import gzip
import tarfile
import io
import os
import shutil

fpath = '/content/RUDY.tar.gz'
VARIANT = 'zero-riscy-a-1-c2'
os.makedirs('/content/labels_N28/RUDY', exist_ok=True)

print("Double decompressing...")
try:
    # First gzip layer
    with gzip.open(fpath, 'rb') as gz1:
        # Second gzip layer → into tarfile
        with gzip.open(gz1, 'rb') as gz2:
            # Read into memory buffer for tarfile
            with tarfile.open(fileobj=gz2, mode='r|') as tar:
                count = 0
                for member in tar:
                    if VARIANT in member.name:
                        print(f"  Found: {member.name} ({member.size/1e6:.2f} MB)")
                        tar.extract(member, path='/content/labels_N28/RUDY')
                        count += 1
                print(f"\n✅ Extracted {count} files!")

except Exception as e:
    print(f"❌ Failed: {e}")

    # Try alternative: decompress to .tar first
    print("\nTrying: decompress to intermediate .tar file...")
    with gzip.open(fpath, 'rb') as gz1:
        inner = gz1.read(512)
        print(f"Inner header hex: {inner[:20].hex()}")
        if inner[:2].hex() == '1f8b':
            print("Confirmed double gzip!")
        elif inner[:5] == b'ustar' or inner[257:262] == b'ustar':
            print("Inner is a tar file directly!")

Double decompressing...
  Found: RUDY/7736-zero-riscy-a-1-c20-u0.85-m2-p8-f0 (0.41 MB)
  Found: RUDY/7209-zero-riscy-a-1-c2-u0.85-m4-p1-f0 (0.41 MB)
  Found: RUDY/7110-zero-riscy-a-1-c2-u0.7-m2-p6-f0 (0.49 MB)
  Found: RUDY/7763-zero-riscy-a-1-c20-u0.9-m2-p4-f0 (0.39 MB)


/tmp/ipykernel_3970/1732292556.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path='/content/labels_N28/RUDY')


  Found: RUDY/7767-zero-riscy-a-1-c20-u0.9-m2-p8-f0 (0.39 MB)
  Found: RUDY/7644-zero-riscy-a-1-c20-u0.7-m3-p4-f0 (0.49 MB)
  Found: RUDY/7140-zero-riscy-a-1-c2-u0.75-m2-p6-f0 (0.46 MB)
  Found: RUDY/7233-zero-riscy-a-1-c2-u0.9-m4-p1-f0 (0.39 MB)
  Found: RUDY/7676-zero-riscy-a-1-c20-u0.75-m3-p4-f0 (0.46 MB)
  Found: RUDY/7170-zero-riscy-a-1-c2-u0.8-m2-p6-f0 (0.44 MB)
  Found: RUDY/7680-zero-riscy-a-1-c20-u0.75-m3-p8-f0 (0.46 MB)
  Found: RUDY/7098-zero-riscy-a-1-c2-u0.7-m1-p2-f0 (0.50 MB)
  Found: RUDY/7200-zero-riscy-a-1-c2-u0.85-m2-p6-f0 (0.41 MB)
  Found: RUDY/7128-zero-riscy-a-1-c2-u0.75-m1-p2-f0 (0.46 MB)
  Found: RUDY/7652-zero-riscy-a-1-c20-u0.7-m4-p4-f0 (0.49 MB)
  Found: RUDY/7743-zero-riscy-a-1-c20-u0.85-m3-p8-f0 (0.41 MB)
  Found: RUDY/7230-zero-riscy-a-1-c2-u0.9-m2-p6-f0 (0.39 MB)
  Found: RUDY/7158-zero-riscy-a-1-c2-u0.8-m1-p2-f0 (0.44 MB)
  Found: RUDY/7656-zero-riscy-a-1-c20-u0.7-m4-p8-f0 (0.49 MB)
  Found: RUDY/7116-zero-riscy-a-1-c2-u0.7-m3-p6-f0 (0.50 MB)
  Found: RU

In [ ]:
import numpy as np
import os
from collections import Counter

graph_base  = '/content/graph_N28'
VARIANT_C2  = 'zero-riscy-a-1-c2'
VARIANT_C20 = 'zero-riscy-a-1-c20'

# Load graph files
node_c2  = np.load(f'{graph_base}/node_attr/{VARIANT_C2}_node_attr.npy',  allow_pickle=True)
pin_c2   = np.load(f'{graph_base}/pin_attr/{VARIANT_C2}_pin_attr.npy',   allow_pickle=True)
net_c2   = np.load(f'{graph_base}/net_attr/{VARIANT_C2}_net_attr.npy',   allow_pickle=True)

node_c20 = np.load(f'{graph_base}/node_attr/{VARIANT_C20}_node_attr.npy', allow_pickle=True)
pin_c20  = np.load(f'{graph_base}/pin_attr/{VARIANT_C20}_pin_attr.npy',  allow_pickle=True)
net_c20  = np.load(f'{graph_base}/net_attr/{VARIANT_C20}_net_attr.npy',  allow_pickle=True)

# Unpack helper (handles list-inside-array issue)
def unpack(arr):
    out = []
    for v in arr:
        if isinstance(v, (list, np.ndarray)):
            out.append(v[0])
        else:
            out.append(v)
    return np.array(out)

def compute_features(pin_data, num_nodes):
    node_ids = unpack(pin_data[2]).astype(int)
    net_ids  = unpack(pin_data[1]).astype(int)

    # Feature 1: degree
    degree = np.zeros(num_nodes, dtype=np.float32)
    for nid in node_ids:
        degree[nid] += 1

    # Feature 2: avg fanout
    net_size = Counter(net_ids.tolist())
    fanout = np.zeros(num_nodes, dtype=np.float32)
    count  = np.zeros(num_nodes, dtype=np.float32)
    for i in range(len(node_ids)):
        nid = node_ids[i]
        fanout[nid] += net_size[net_ids[i]]
        count[nid]  += 1

    # FIX: explicit out= array so nodes with count==0 get a real 0,
    # not uninitialized garbage from np.empty_like.
    avg_fanout = np.zeros(num_nodes, dtype=np.float32)
    avg_fanout = np.divide(fanout, count, out=avg_fanout, where=count > 0)

    return np.stack([degree, avg_fanout], axis=1)

print("Computing features (takes ~30 sec)...")
feats_c2  = compute_features(pin_c2,  node_c2.shape[1])
feats_c20 = compute_features(pin_c20, node_c20.shape[1])

print(f"✅ C2  node features : {feats_c2.shape}")
print(f"✅ C20 node features : {feats_c20.shape}")
print(f"\nC2  degree  : min={feats_c2[:,0].min():.1f} max={feats_c2[:,0].max():.1f} mean={feats_c2[:,0].mean():.2f}")
print(f"C2  fanout  : min={feats_c2[:,1].min():.1f} max={feats_c2[:,1].max():.1f} mean={feats_c2[:,1].mean():.2f}")
print(f"C20 degree  : min={feats_c20[:,0].min():.1f} max={feats_c20[:,0].max():.1f} mean={feats_c20[:,0].mean():.2f}")
print(f"C20 fanout  : min={feats_c20[:,1].min():.1f} max={feats_c20[:,1].max():.1f} mean={feats_c20[:,1].mean():.2f}")

Computing features (takes ~30 sec)...
✅ C2  node features : (42102, 2)
✅ C20 node features : (41405, 2)

C2  degree  : min=1.0 max=30.0 mean=3.99
C2  fanout  : min=1.0 max=809.0 mean=43.58
C20 degree  : min=1.0 max=30.0 mean=4.03
C20 fanout  : min=1.0 max=809.0 mean=44.52


In [9]:
!pip install torch-geometric -q
print("✅ Done!")

✅ Done!


In [10]:
import numpy as np
import os
import cv2   # OpenCV

label_path = "/content/labels_N28/RUDY/RUDY"
TARGET_SIZE = (256, 256)

def load_labels(folder, target_config):
    label_list = []

    for f in os.listdir(folder):
        parts = f.split('-')
        if len(parts) > 5 and parts[5] == target_config:
            arr = np.load(os.path.join(folder, f))

            # Resize to fixed size
            arr_resized = cv2.resize(arr, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)
            label_list.append(arr_resized)

    print(f"{target_config} files found:", len(label_list))

    labels = np.stack(label_list)
    return labels

print("Loading C2 labels...")
c2_labels = load_labels(label_path, "c2")
print("C2 labels shape:", c2_labels.shape)

print("Loading C20 labels...")
c20_labels = load_labels(label_path, "c20")
print("C20 labels shape:", c20_labels.shape)


Loading C2 labels...
c2 files found: 264
C2 labels shape: (264, 256, 256)
Loading C20 labels...
c20 files found: 271
C20 labels shape: (271, 256, 256)


In [ ]:
import torch
from torch_geometric.data import Data
import numpy as np
import re
from collections import defaultdict

def build_edge_index(pin_data, num_nodes):
    net_ids  = unpack(pin_data[1]).astype(int)
    node_ids = unpack(pin_data[2]).astype(int)
    net_to_nodes = defaultdict(set)
    for i in range(len(net_ids)):
        net_to_nodes[net_ids[i]].add(node_ids[i])
    edges = []
    for net, nodes in net_to_nodes.items():
        nodes = list(nodes)
        if len(nodes) < 2:
            continue
        for i in range(1, len(nodes)):
            edges.append([nodes[0], nodes[i]])
            edges.append([nodes[i], nodes[0]])  # undirected
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

print("Building edge_index for C2 ...")
ei_c2  = build_edge_index(pin_c2,  node_c2.shape[1])
print(f"✅ C2  edge_index : {ei_c2.shape}")

print("Building edge_index for C20 ...")
ei_c20 = build_edge_index(pin_c20, node_c20.shape[1])
print(f"✅ C20 edge_index : {ei_c20.shape}")

# ============================================================
# Node-feature scaler -- fit on C2's structural features. This is
# a single fixed netlist (one array of per-node degree/avg_fanout,
# identical across all 264 configs), NOT per-config data, so there
# is no train/val split to worry about here.
# ============================================================
feat_mean = feats_c2.mean(axis=0)
feat_std  = feats_c2.std(axis=0) + 1e-6

feats_c2_norm  = (feats_c2  - feat_mean) / feat_std
feats_c20_norm = (feats_c20 - feat_mean) / feat_std

# ============================================================
# Config params (u, m, p, f) parsed from RUDY filenames.
# Real filename format:
#   7097-zero-riscy-a-1-c2-u0.7-m1-p1-f0.npy
# ============================================================
CONFIG_RE = re.compile(r'u([\d.]+)-m(\d+)-p(\d+)-f(\d+)')

def parse_n28_config(fname):
    match = CONFIG_RE.search(fname)
    if not match:
        raise ValueError(f"Could not parse config from filename: {fname}")
    u, m, p, f = match.groups()
    return np.array([float(u), float(m), float(p), float(f)], dtype=np.float32)

def load_labels_with_filenames(folder, target_config):
    """Same filter logic as before, but ALSO keeps filenames in lockstep
    with the label array so we can parse each config's params."""
    fnames, label_list = [], []
    for f in sorted(os.listdir(folder)):
        parts = f.split('-')
        if len(parts) > 5 and parts[5] == target_config:
            arr = np.load(os.path.join(folder, f))
            arr_resized = cv2.resize(arr, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)
            label_list.append(arr_resized)
            fnames.append(f)
    return fnames, np.stack(label_list)

c2_fnames,  c2_labels  = load_labels_with_filenames(label_path, "c2")
c20_fnames, c20_labels = load_labels_with_filenames(label_path, "c20")
print(f"\nC2  : {len(c2_fnames)} configs   C20 : {len(c20_fnames)} configs")

c2_mean_label  = c2_labels.mean(axis=(1, 2))
c20_mean_label = c20_labels.mean(axis=(1, 2))

c2_configs  = np.stack([parse_n28_config(f) for f in c2_fnames])   # (264, 4)
c20_configs = np.stack([parse_n28_config(f) for f in c20_fnames])  # (271, 4)

# ============================================================
# Train/val split decided BEFORE fitting the config-param scaler,
# so cfg normalization has zero leakage from validation configs.
# ============================================================
n_c2 = c2_configs.shape[0]
rng = np.random.default_rng(42)
perm = rng.permutation(n_c2)
n_train = int(0.8 * n_c2)
c2_train_idx, c2_val_idx = perm[:n_train], perm[n_train:]

cfg_mean = c2_configs[c2_train_idx].mean(axis=0)
cfg_std  = c2_configs[c2_train_idx].std(axis=0) + 1e-6

c2_configs_norm  = (c2_configs  - cfg_mean) / cfg_std
c20_configs_norm = (c20_configs - cfg_mean) / cfg_std

# ============================================================
# ARCHITECTURE FIX: x = structural features ONLY (degree, avg_fanout,
# dim=2). Config params live in a separate `cfg` attribute per graph,
# NOT concatenated into x. This keeps the GNN backbone's input
# dimension identical across N28 and N14 (both = 2), which the
# frozen-backbone + LoRA cross-domain transfer requires.
# ============================================================
def build_per_config_dataset(base_feats_norm, edge_index, configs_norm, labels):
    x = torch.tensor(base_feats_norm, dtype=torch.float)   # shared across all configs of this domain
    data_list = []
    for i in range(configs_norm.shape[0]):
        data_list.append(Data(
            x          = x,
            edge_index = edge_index,
            cfg        = torch.tensor(configs_norm[i], dtype=torch.float).unsqueeze(0),  # (1, 4)
            y          = torch.tensor([labels[i]], dtype=torch.float)
        ))
    return data_list

data_c2_list  = build_per_config_dataset(feats_c2_norm,  ei_c2,  c2_configs_norm,  c2_mean_label)
data_c20_list = build_per_config_dataset(feats_c20_norm, ei_c20, c20_configs_norm, c20_mean_label)

print(f"\n✅ data_c2_list  : {len(data_c2_list)} graphs, x shape {data_c2_list[0].x.shape}, cfg shape {data_c2_list[0].cfg.shape}")
print(f"✅ data_c20_list : {len(data_c20_list)} graphs, x shape {data_c20_list[0].x.shape}, cfg shape {data_c20_list[0].cfg.shape}")
print(f"✅ C2 split: {len(c2_train_idx)} train / {len(c2_val_idx)} val")
print(f"\n🎯 GNN backbone in_channels = {data_c2_list[0].x.shape[1]} (structural only; cfg handled separately)")

Building edge_index for C2 ...
✅ C2  edge_index : torch.Size([2, 242230])
Building edge_index for C20 ...
✅ C20 edge_index : torch.Size([2, 240714])

C2  : 264 configs   C20 : 271 configs

✅ data_c2_list  : 264 graphs, each x shape torch.Size([42102, 6])
✅ data_c20_list : 271 graphs, each x shape torch.Size([41405, 6])

🎯 Preprocessing DONE (per-config). GNN in_channels = 6 (2 base + 4 config)


In [ ]:
import numpy as np
import os
import torch

save_dir = '/content/drive/MyDrive/PEDA_Processed'
os.makedirs(save_dir, exist_ok=True)

torch.save(data_c2_list,  f'{save_dir}/data_c2_raw.pt')
torch.save(data_c20_list, f'{save_dir}/data_c20_raw.pt')

# Node-feature scaler (structural, fit on C2's single netlist)
np.save(f'{save_dir}/scaler_mean.npy', feat_mean)
np.save(f'{save_dir}/scaler_std.npy',  feat_std)

# Config-param scaler (fit on C2 TRAIN subset only -- no leakage)
np.save(f'{save_dir}/config_mean.npy', cfg_mean)
np.save(f'{save_dir}/config_std.npy',  cfg_std)

# Split indices -- save these so training uses the SAME split,
# not a fresh random one each run.
np.save(f'{save_dir}/c2_train_idx.npy', c2_train_idx)
np.save(f'{save_dir}/c2_val_idx.npy',   c2_val_idx)

print(f"✅ Saved data_c2_raw.pt  ({len(data_c2_list)} graphs, labels unnormalized)")
print(f"✅ Saved data_c20_raw.pt ({len(data_c20_list)} graphs, labels unnormalized)")
print(f"✅ Saved node-feature scaler (scaler_mean/std.npy)")
print(f"✅ Saved config-param scaler, fit on {len(c2_train_idx)} train configs only")
print(f"✅ Saved c2_train_idx.npy ({len(c2_train_idx)}) / c2_val_idx.npy ({len(c2_val_idx)})")

✅ Saved data_c2_raw.pt  (264 graphs, labels unnormalized)
✅ Saved data_c20_raw.pt (271 graphs, labels unnormalized)
✅ Saved node-feature scaler (scaler_mean/std.npy)
✅ Saved config-param scaler (config_mean/std.npy)

🎯 Research-grade correct: per-config features, C2-only scaling, raw labels, no leakage.


In [ ]:
import torch
import numpy as np

print("=== FINAL PREPROCESSING VERIFICATION (C2/C20) ===\n")

save_dir = '/content/drive/MyDrive/PEDA_Processed'
mean_c2   = np.load(f'{save_dir}/scaler_mean.npy')
std_c2    = np.load(f'{save_dir}/scaler_std.npy')
cfg_mean  = np.load(f'{save_dir}/config_mean.npy')
cfg_std   = np.load(f'{save_dir}/config_std.npy')
c2_train_idx = np.load(f'{save_dir}/c2_train_idx.npy')
c2_val_idx   = np.load(f'{save_dir}/c2_val_idx.npy')

def summarize(name, data_list):
    y_all = torch.cat([d.y for d in data_list])
    print(f"{name}:")
    print(f"  Configs      : {len(data_list)}")
    print(f"  Nodes/graph  : {data_list[0].x.shape[0]}")
    print(f"  x  (struct)  : {data_list[0].x.shape[1]} dims (degree, avg_fanout)")
    print(f"  cfg (config) : {data_list[0].cfg.shape[1]} dims (u, m, p, f)")
    print(f"  Edges/graph  : {data_list[0].edge_index.shape[1]}")
    print(f"  Label range  : {y_all.min():.6f} ~ {y_all.max():.6f}  (raw, unnormalized)")

summarize("data_c2_list  (SOURCE - N28 c2)", data_c2_list)
print()
summarize("data_c20_list (TARGET - N28 c20)", data_c20_list)

print(f"\n=== LEAKAGE CHECK ===")
print(f"Node-feature scaler: fit on C2 structural array (not per-config, no split needed)")
print(f"Config-param scaler: fit on C2 TRAIN subset only : {len(c2_train_idx)} configs")
print(f"C20 uses C2's train-fit scalers (both node + config)  : yes")
print(f"Node-feature mean used : {mean_c2}")
print(f"Node-feature std  used : {std_c2}")
print(f"Config-param  mean used : {cfg_mean}")
print(f"Config-param  std  used : {cfg_std}")

print(f"\n=== ACTUAL DATA SPLIT ===")
print(f"C2  train : {len(c2_train_idx)} configs")
print(f"C2  val   : {len(c2_val_idx)} configs")
print(f"C20 test  : {len(data_c20_list)} configs (cross-domain, same tech node)")

print(f"\n✅ PREPROCESSING VERIFIED — split is real and saved, scalers are train-only")

=== FINAL PREPROCESSING VERIFICATION (C2/C20) ===

data_c2_list  (SOURCE - N28 c2):
  Configs      : 264
  Nodes/graph  : 42102
  Features     : 6 (2 base: degree, avg_fanout + 4 config: u, m, p, f)
  Edges/graph  : 242230
  Label range  : 0.001259 ~ 0.002377  (raw, unnormalized)

data_c20_list (TARGET - N28 c20):
  Configs      : 271
  Nodes/graph  : 41405
  Features     : 6 (2 base: degree, avg_fanout + 4 config: u, m, p, f)
  Edges/graph  : 240714
  Label range  : 0.001315 ~ 0.006693  (raw, unnormalized)

=== LEAKAGE CHECK ===
Node-feature scaler fit on C2 only  : yes
Config-param scaler fit on C2 only  : yes
C20 uses C2 scaler stats (both)     : yes
Node-feature mean used : [ 3.9944184 43.580032 ]
Node-feature std  used : [ 1.1122626 84.87292  ]
Config-param  mean used : [0.7977272  2.2424242  4.560606   0.45454547]
Config-param  std  used : [0.06986656 1.0453458  2.2970939  0.49793026]

=== DATA SPLIT PLAN ===
C2  train : 211 configs (80%)
C2  val   : 53   configs (20%)
C20 test  

In [14]:
##############################################################N14
import tarfile
import os

# Extract N14 graph_information directly from Drive
tar_path   = '/content/drive/MyDrive/CircuitNet/CircuitNet-N14/graph_features/graph_information.tar.gz'
extract_to = '/content/graph_N14/'
os.makedirs(extract_to, exist_ok=True)

print("Extracting N14 graph_information...")
with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=extract_to)
print("✅ Extracted!")

# Check what designs we have
print("\nN14 node_attr files:")
node_files = os.listdir(f'{extract_to}/node_attr/')
for f in node_files[:10]:
    print(f"  {f}")
print(f"\nTotal designs: {len(node_files)}")

Extracting N14 graph_information...


/tmp/ipykernel_3970/857824064.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


✅ Extracted!

N14 node_attr files:
  san_nvdla-large_5_netlist_flatten_node_attr.npy
  san_zero-riscy_20_netlist_flatten_node_attr.npy
  san_nvdla-small_2_netlist_flatten_node_attr.npy
  san_Vortex-large_5_netlist_flatten_node_attr.npy
  san_nvdla-large_2_netlist_flatten_node_attr.npy
  san_Vrotex-small_2_netlist_flatten_node_attr.npy
  san_RISCY_5_netlist_flatten_node_attr.npy
  san_RISCY-FPU_20_netlist_flatten_node_attr.npy
  san_zero-riscy_2_netlist_flatten_node_attr.npy
  san_RISCY-FPU_5_netlist_flatten_node_attr.npy

Total designs: 19


In [15]:
import shutil
import os
import gzip

src = '/content/drive/MyDrive/CircuitNet/CircuitNet-N14/routability_features/RUDY/RUDY.tar.gz'
dst = '/content/RUDY_N14.tar.gz'

if os.path.exists(dst):
    print(f"Already exists! Size: {os.path.getsize(dst)/(1024*1024):.1f} MB")
else:
    print("Copying N14 RUDY.tar.gz (~3.8GB, takes ~10 mins)...")
    shutil.copy2(src, dst)
    print(f"Done! Size: {os.path.getsize(dst)/(1024*1024):.1f} MB")

print("\nChecking compression format...")
with gzip.open(dst, 'rb') as gz:
    inner = gz.read(2)
    if inner.hex() == '1f8b':
        print("Double gzip confirmed!")
    else:
        print(f"Single gzip. Inner header: {inner.hex()}")

Copying N14 RUDY.tar.gz (~3.8GB, takes ~10 mins)...
Done! Size: 3881.6 MB

Checking compression format...
Single gzip. Inner header: 5255


In [16]:
import gzip
import tarfile
import os
import shutil
import numpy as np

dst = '/content/RUDY_N14.tar.gz'
VARIANT = 'zero-riscy'
os.makedirs('/content/labels_N14/RUDY', exist_ok=True)

print(f"Extracting only {VARIANT} files (single gzip)...")
with tarfile.open(dst, 'r:gz') as tar:
    count = 0
    for member in tar:
        if VARIANT in member.name:
            tar.extract(member, path='/content/labels_N14/RUDY')
            count += 1
            if count % 50 == 0:
                print(f"  Extracted {count} files so far...")
print(f"\nExtracted {count} files!")

# Delete tar immediately!
os.remove(dst)
print("Deleted tar! Freed 3.8GB!")

# Check storage
total, used, free = shutil.disk_usage('/content')
print(f"Storage: {used/1e9:.1f}GB used / {free/1e9:.1f}GB free")

# Check extracted files
files = os.listdir('/content/labels_N14/RUDY/RUDY')
print(f"\nTotal zero-riscy N14 files: {len(files)}")
sample = np.load(f'/content/labels_N14/RUDY/RUDY/{files[0]}')
print(f"Sample shape : {sample.shape}")
print(f"Sample range : min={sample.min():.4f} max={sample.max():.4f}")

Extracting only zero-riscy files (single gzip)...


/tmp/ipykernel_3970/3053706489.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path='/content/labels_N14/RUDY')


  Extracted 50 files so far...
  Extracted 100 files so far...
  Extracted 150 files so far...
  Extracted 200 files so far...
  Extracted 250 files so far...
  Extracted 300 files so far...
  Extracted 350 files so far...
  Extracted 400 files so far...
  Extracted 450 files so far...
  Extracted 500 files so far...
  Extracted 550 files so far...
  Extracted 600 files so far...
  Extracted 650 files so far...
  Extracted 700 files so far...
  Extracted 750 files so far...
  Extracted 800 files so far...
  Extracted 850 files so far...
  Extracted 900 files so far...
  Extracted 950 files so far...
  Extracted 1000 files so far...
  Extracted 1050 files so far...
  Extracted 1100 files so far...
  Extracted 1150 files so far...
  Extracted 1200 files so far...
  Extracted 1250 files so far...
  Extracted 1300 files so far...
  Extracted 1350 files so far...
  Extracted 1400 files so far...
  Extracted 1450 files so far...
  Extracted 1500 files so far...
  Extracted 1550 files so far.

In [ ]:
import tarfile
import numpy as np
import torch
from collections import Counter, defaultdict
from PIL import Image
import os, gc

# -- Step 1: Re-extract N14 graph --
tar_path   = '/content/drive/MyDrive/CircuitNet/CircuitNet-N14/graph_features/graph_information.tar.gz'
extract_to = '/content/graph_N14/'
os.makedirs(extract_to, exist_ok=True)

print("Re-extracting N14 graph...")
with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=extract_to)
print("✅ Done!")

VARIANT_N14 = 'san_zero-riscy_5_netlist_flatten'
node_n14 = np.load(f'{extract_to}/node_attr/{VARIANT_N14}_node_attr.npy', allow_pickle=True)
pin_n14  = np.load(f'{extract_to}/pin_attr/{VARIANT_N14}_pin_attr.npy',  allow_pickle=True)
print(f"✅ Loaded: {VARIANT_N14}  nodes={node_n14.shape[1]}  pins={pin_n14.shape[1]}")

def unpack(arr):
    return np.array([v[0] if isinstance(v, (list, np.ndarray)) else v for v in arr])

def unpack_int(arr):
    return np.array([int(v[0]) if isinstance(v, (list, np.ndarray)) else int(v) for v in arr])

def compute_features(pin_data, num_nodes):
    node_ids = unpack_int(pin_data[2])
    net_ids  = unpack(pin_data[1]).astype(int)
    degree   = np.zeros(num_nodes, dtype=np.float32)
    for nid in node_ids:
        degree[nid] += 1
    net_size = Counter(net_ids.tolist())
    fanout   = np.zeros(num_nodes, dtype=np.float32)
    count    = np.zeros(num_nodes, dtype=np.float32)
    for i in range(len(node_ids)):
        nid = node_ids[i]
        fanout[nid] += net_size[net_ids[i]]
        count[nid]  += 1
    # FIX: explicit out= array, same as N28's compute_features
    avg_fanout = np.zeros(num_nodes, dtype=np.float32)
    avg_fanout = np.divide(fanout, count, out=avg_fanout, where=count > 0)
    return np.stack([degree, avg_fanout], axis=1)

def build_edge_index(pin_data, num_nodes):
    net_ids  = unpack(pin_data[1]).astype(int)
    node_ids = unpack_int(pin_data[2])
    net_to_nodes = defaultdict(set)
    for i in range(len(net_ids)):
        net_to_nodes[net_ids[i]].add(node_ids[i])
    edges = []
    for net, nodes in net_to_nodes.items():
        nodes = list(nodes)
        if len(nodes) < 2:
            continue
        for i in range(1, len(nodes)):
            edges.append([nodes[0], nodes[i]])
            edges.append([nodes[i], nodes[0]])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

feats_n14 = compute_features(pin_n14, node_n14.shape[1])
ei_n14    = build_edge_index(pin_n14, node_n14.shape[1])
print(f"✅ Features: {feats_n14.shape}   edge_index: {tuple(ei_n14.shape)}")

del node_n14, pin_n14
gc.collect()

# Use N28's structural scaler -- keeps N14's x on the same normalization
# convention as the frozen backbone was trained on.
mean_c2 = np.load('/content/drive/MyDrive/PEDA_Processed/scaler_mean.npy')
std_c2  = np.load('/content/drive/MyDrive/PEDA_Processed/scaler_std.npy')
feats_n14_norm = (feats_n14 - mean_c2) / std_c2
del feats_n14
print(f"✅ Normalized base features: min={feats_n14_norm.min():.3f} max={feats_n14_norm.max():.3f}")

def parse_n14_config(fname):
    parts = fname.replace('.npy', '').split('_')
    freq = float(parts[parts.index('freq') + 1])
    mp   = float(parts[parts.index('mp')   + 1])
    fpu  = float(parts[parts.index('fpu')  + 1])
    fpa  = float(parts[parts.index('fpa')  + 1])
    p    = float(parts[parts.index('p')    + 1])
    return np.array([freq, mp, fpu, fpa, p], dtype=np.float32)

rudy_base = '/content/labels_N14/RUDY/RUDY'
files_n14 = sorted(os.listdir(rudy_base))
print(f"\nProcessing {len(files_n14)} RUDY label files (memory-safe)...")

TARGET = (256, 256)
n14_mean_label = np.zeros(len(files_n14), dtype=np.float32)
n14_configs    = np.zeros((len(files_n14), 5), dtype=np.float32)

for i, f in enumerate(files_n14):
    arr = np.load(f'{rudy_base}/{f}').astype(np.float32)
    img = Image.fromarray(arr).resize(TARGET, Image.BILINEAR)
    n14_mean_label[i] = np.array(img).mean()
    n14_configs[i]    = parse_n14_config(f)
    del arr, img
    if (i + 1) % 500 == 0:
        gc.collect()
        print(f"  Processed {i+1}/{len(files_n14)}...")

print(f"✅ Labels: {n14_mean_label.shape}   Configs: {n14_configs.shape}")

# ============================================================
# Split N14 into adaptation-train / val BEFORE fitting its
# config-param scaler -- same no-leakage rule as C2.
# ============================================================
n_n14 = n14_configs.shape[0]
rng2 = np.random.default_rng(42)
perm2 = rng2.permutation(n_n14)
n_train14 = int(0.8 * n_n14)
n14_train_idx, n14_val_idx = perm2[:n_train14], perm2[n_train14:]

cfg_mean_n14 = n14_configs[n14_train_idx].mean(axis=0)
cfg_std_n14  = n14_configs[n14_train_idx].std(axis=0) + 1e-6
n14_configs_norm = (n14_configs - cfg_mean_n14) / cfg_std_n14

save_dir = '/content/drive/MyDrive/PEDA_Processed'
torch.save(torch.tensor(feats_n14_norm, dtype=torch.float), f'{save_dir}/n14_base_x.pt')
torch.save(ei_n14, f'{save_dir}/n14_edge_index.pt')
np.save(f'{save_dir}/n14_configs_norm.npy', n14_configs_norm)
np.save(f'{save_dir}/n14_labels_raw.npy',   n14_mean_label)
np.save(f'{save_dir}/n14_config_mean.npy',  cfg_mean_n14)
np.save(f'{save_dir}/n14_config_std.npy',   cfg_std_n14)
np.save(f'{save_dir}/n14_train_idx.npy',    n14_train_idx)
np.save(f'{save_dir}/n14_val_idx.npy',      n14_val_idx)

print(f"\n✅ Saved compact N14 representation:")
print(f"   n14_base_x.pt        : {feats_n14_norm.shape}  (shared across all configs)")
print(f"   n14_edge_index.pt    : {tuple(ei_n14.shape)}")
print(f"   n14_configs_norm.npy : {n14_configs_norm.shape}")
print(f"   n14_labels_raw.npy   : {n14_mean_label.shape}")
print(f"   n14 split            : {len(n14_train_idx)} train / {len(n14_val_idx)} val")

# ============================================================
# ARCHITECTURE FIX: x and cfg are returned SEPARATELY, not
# concatenated. This keeps N14's x at 2-dim, matching N28's x,
# so the frozen backbone accepts both domains identically.
# ============================================================
def make_n14_data(i, base_x, edge_index, configs_norm, labels_raw):
    """Build ONE N14 Data object on demand. x stays 2-dim (structural);
    cfg (5-dim) is a separate field consumed by the domain-specific head."""
    from torch_geometric.data import Data
    return Data(
        x          = base_x,
        edge_index = edge_index,
        cfg        = torch.tensor(configs_norm[i], dtype=torch.float).unsqueeze(0),
        y          = torch.tensor([labels_raw[i]], dtype=torch.float)
    )

print(f"\nSanity check -- building one on-demand sample:")
sample = make_n14_data(0, torch.tensor(feats_n14_norm, dtype=torch.float), ei_n14, n14_configs_norm, n14_mean_label)
print(f"  data_n14[0] : x={sample.x.shape}  cfg={sample.cfg.shape}  edge_index={tuple(sample.edge_index.shape)}  y={sample.y}")
print(f"\n🎯 N14 x dim = {sample.x.shape[1]} (matches N28 x dim = 2)  |  N14 cfg dim = {sample.cfg.shape[1]} (domain-specific)")

Re-extracting N14 graph...


/tmp/ipykernel_3970/1370414067.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


✅ Done!
✅ Loaded: san_zero-riscy_5_netlist_flatten  nodes=34983  pins=136946
✅ Features: (34983, 2)   edge_index: (2, 202756)
✅ Normalized base features: min=-1.793 max=20.684

Processing 3441 RUDY label files (memory-safe)...
  Processed 500/3441...
  Processed 1000/3441...
  Processed 1500/3441...
  Processed 2000/3441...
  Processed 2500/3441...
  Processed 3000/3441...
✅ Labels: (3441,)   Configs: (3441, 5)

✅ Saved compact N14 representation (no duplication):
   n14_base_x.pt        : (34983, 2)  (shared across all configs)
   n14_edge_index.pt    : (2, 202756)
   n14_configs_norm.npy : (3441, 5)
   n14_labels_raw.npy   : (3441,)

🎯 3441 configs saved compactly -- reconstruct each graph on demand, not all at once.

Sanity check -- building one on-demand sample:
  data_n14[0] : x=torch.Size([34983, 7])  edge_index=(2, 202756)  y=tensor([0.0051])


In [ ]:
import torch
import numpy as np

save_dir = '/content/drive/MyDrive/PEDA_Processed'

data_c2_list  = torch.load(f'{save_dir}/data_c2_raw.pt',  weights_only=False)
data_c20_list = torch.load(f'{save_dir}/data_c20_raw.pt', weights_only=False)

# N14 uses the compact format now -- no data_n14_raw.pt anymore
n14_base_x     = torch.load(f'{save_dir}/n14_base_x.pt')
n14_edge_index = torch.load(f'{save_dir}/n14_edge_index.pt')
n14_labels     = np.load(f'{save_dir}/n14_labels_raw.npy')

print("=" * 50)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 50)

def summarize(name, data_list):
    y_all = torch.cat([d.y for d in data_list])
    print(f"\n {name}")
    print(f"   Configs  : {len(data_list)}")
    print(f"   Nodes    : {data_list[0].x.shape[0]}")
    print(f"   Edges    : {data_list[0].edge_index.shape[1]}")
    print(f"   Features : {data_list[0].x.shape[1]}")
    print(f"   y range  : {y_all.min():.6f} ~ {y_all.max():.6f}")

summarize("N28 C2  (SOURCE - train/val)", data_c2_list)
summarize("N28 C20 (SAME-DOMAIN TARGET - test)", data_c20_list)

y_n14 = torch.tensor(n14_labels)
print(f"\n N14     (CROSS-DOMAIN TARGET - test)")
print(f"   Configs  : {len(n14_labels)}")
print(f"   Nodes    : {n14_base_x.shape[0]}")
print(f"   Edges    : {n14_edge_index.shape[1]}")
print(f"   Structural x : {n14_base_x.shape[1]} dims  |  Config cfg : 5 dims (kept separate)")
print(f"   y range  : {y_n14.min():.6f} ~ {y_n14.max():.6f}")

print(f"\n{'=' * 50}")
print(f"LEAKAGE CHECK")
print(f"{'=' * 50}")
mean_c2 = np.load(f'{save_dir}/scaler_mean.npy')
std_c2  = np.load(f'{save_dir}/scaler_std.npy')
print(f"   Node-feature scaler fit on C2 only : yes")
print(f"   mean = {mean_c2}")
print(f"   std  = {std_c2}")

print(f"\n{'=' * 50}")
print(f"DATA SPLIT PLAN")
print(f"{'=' * 50}")
n_c2 = len(data_c2_list)
c2_train = int(0.8 * n_c2)
c2_val   = n_c2 - c2_train
print(f"   C2  train : {c2_train} configs (80%)")
print(f"   C2  val   : {c2_val} configs (20%)")
print(f"   C20 test  : {len(data_c20_list)} configs (same-domain)")
print(f"   N14 test  : {len(n14_labels)} configs (cross-domain)")

print(f"\n ALL 3 DATASETS READY!")
print(f" NEXT \u2192 Build GNN model!")

FINAL PREPROCESSING SUMMARY

 N28 C2  (SOURCE - train/val)
   Configs  : 264
   Nodes    : 42102
   Edges    : 242230
   Features : 6
   y range  : 0.001259 ~ 0.002377

 N28 C20 (SAME-DOMAIN TARGET - test)
   Configs  : 271
   Nodes    : 41405
   Edges    : 240714
   Features : 6
   y range  : 0.001315 ~ 0.006693

 N14     (CROSS-DOMAIN TARGET - test)
   Configs  : 3441
   Nodes    : 34983
   Edges    : 202756
   Features : 7  (built on demand, not stored duplicated)
   y range  : 0.002962 ~ 0.006666

LEAKAGE CHECK
   Node-feature scaler fit on C2 only : yes
   mean = [ 3.9944184 43.580032 ]
   std  = [ 1.1122626 84.87292  ]

DATA SPLIT PLAN
   C2  train : 211 configs (80%)
   C2  val   : 53 configs (20%)
   C20 test  : 271 configs (same-domain)
   N14 test  : 3441 configs (cross-domain)

 ALL 3 DATASETS READY!
 NEXT → Build GNN model!


In [ ]:
import torch
import numpy as np

save_dir = '/content/drive/MyDrive/PEDA_Processed'

# ============================================
# LOAD C2 / C20
# ============================================

data_c2_list = torch.load(
    f'{save_dir}/data_c2_raw.pt',
    weights_only=False
)

data_c20_list = torch.load(
    f'{save_dir}/data_c20_raw.pt',
    weights_only=False
)


# ============================================
# LOAD N14 COMPACT REPRESENTATION
# ============================================

n14_base_x = torch.load(
    f'{save_dir}/n14_base_x.pt'
)

n14_edge_index = torch.load(
    f'{save_dir}/n14_edge_index.pt'
)

n14_configs = np.load(
    f'{save_dir}/n14_configs_norm.npy'
)

n14_labels = np.load(
    f'{save_dir}/n14_labels_raw.npy'
)


# ============================================
# AUDIT C2 / C20
# ============================================

def audit(name, data_list):

    x_all = torch.cat([d.x for d in data_list], dim=0)
    y_all = torch.cat([d.y for d in data_list])

    x_nan = torch.isnan(x_all).any().item()
    x_inf = torch.isinf(x_all).any().item()

    y_nan = torch.isnan(y_all).any().item()
    y_inf = torch.isinf(y_all).any().item()

    ei_ok = all(
        d.edge_index.min().item() >= 0
        and d.edge_index.max().item() < d.x.shape[0]
        for d in data_list
    )

    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")

    print(f"  configs      : {len(data_list)}")
    print(f"  x shape      : {data_list[0].x.shape}  (per graph)")

    print(f"  x NaN        : {x_nan}  {'❌' if x_nan else '✅'}")
    print(f"  x Inf        : {x_inf}  {'❌' if x_inf else '✅'}")
    print(f"  x range      : {x_all.min():.4f} ~ {x_all.max():.4f}")

    print(f"  y NaN        : {y_nan}  {'❌' if y_nan else '✅'}")
    print(f"  y Inf        : {y_inf}  {'❌' if y_inf else '✅'}")
    print(f"  y range      : {y_all.min():.6f} ~ {y_all.max():.6f}")

    print(f"  valid edges  : {'✅' if ei_ok else '❌'}")


audit("N28 C2  (SOURCE)", data_c2_list)
audit("N28 C20 (SAME-DOMAIN TARGET)", data_c20_list)


# ============================================
# AUDIT N14 COMPACT FORMAT
# ============================================

print(f"\n{'='*40}")
print("N14     (CROSS-DOMAIN TARGET)")
print(f"{'='*40}")

n14_x_nan = torch.isnan(n14_base_x).any().item()
n14_x_inf = torch.isinf(n14_base_x).any().item()

n14_cfg_nan = np.isnan(n14_configs).any()
n14_cfg_inf = np.isinf(n14_configs).any()

n14_y_nan = np.isnan(n14_labels).any()
n14_y_inf = np.isinf(n14_labels).any()

n14_edge_ok = (
    n14_edge_index.min().item() >= 0
    and n14_edge_index.max().item() < n14_base_x.shape[0]
)

print(f"  configs      : {len(n14_labels)}")
print(f"  base x shape : {n14_base_x.shape}")
print(f"  config shape : {n14_configs.shape}")

print(f"  x NaN        : {n14_x_nan}  {'❌' if n14_x_nan else '✅'}")
print(f"  x Inf        : {n14_x_inf}  {'❌' if n14_x_inf else '✅'}")
print(f"  x range      : {n14_base_x.min():.4f} ~ {n14_base_x.max():.4f}")

print(f"  config NaN   : {n14_cfg_nan}  {'❌' if n14_cfg_nan else '✅'}")
print(f"  config Inf   : {n14_cfg_inf}  {'❌' if n14_cfg_inf else '✅'}")

print(f"  y NaN        : {n14_y_nan}  {'❌' if n14_y_nan else '✅'}")
print(f"  y Inf        : {n14_y_inf}  {'❌' if n14_y_inf else '✅'}")
print(f"  y range      : {n14_labels.min():.6f} ~ {n14_labels.max():.6f}")

print(f"  valid edges  : {'✅' if n14_edge_ok else '❌'}")


# ============================================
# SCALER CHECK
# ============================================

print(f"\n{'='*40}")
print("SCALER CHECK")
print(f"{'='*40}")

mean_c2 = np.load(
    f'{save_dir}/scaler_mean.npy'
)

std_c2 = np.load(
    f'{save_dir}/scaler_std.npy'
)

print(f"  mean : {mean_c2}")
print(f"  std  : {std_c2}")

scaler_bad = (
    np.isnan(mean_c2).any()
    or np.isnan(std_c2).any()
    or np.isinf(mean_c2).any()
    or np.isinf(std_c2).any()
    or np.any(std_c2 == 0)
)

print(f"  NaN/Inf/zero std : {scaler_bad} {'❌' if scaler_bad else '✅'}")


print(f"\n{'='*40}")
print("🎯 PREPROCESSING FULLY VERIFIED!")
print(f"{'='*40}")

# ///////
# ============================================================
# CROSS-DOMAIN DIMENSION ASSERTIONS
# ============================================================
print(f"\n{'='*40}")
print("CROSS-DOMAIN DIMENSION CHECK")
print(f"{'='*40}")

assert data_c2_list[0].x.shape[1] == 2,  "N28 C2 x should be 2-dim (structural only)"
assert data_c20_list[0].x.shape[1] == 2, "N28 C20 x should be 2-dim (structural only)"
assert n14_base_x.shape[1] == 2,         "N14 x should be 2-dim (structural only)"

assert data_c2_list[0].cfg.shape[1] == 4,  "N28 cfg should be 4-dim (u,m,p,f)"
assert data_c20_list[0].cfg.shape[1] == 4, "N28 C20 cfg should be 4-dim (u,m,p,f)"
assert n14_configs.shape[1] == 5,          "N14 cfg should be 5-dim (freq,mp,fpu,fpa,p)"

print("✅ N28 x dim = 2, N14 x dim = 2 -- structural dims MATCH")
print("✅ N28 cfg dim = 4, N14 cfg dim = 5 -- config dims stay domain-specific (expected)")
print("✅ Frozen GNN backbone can accept both N28 and N14 graphs")



N28 C2  (SOURCE)
  configs      : 264
  x shape      : torch.Size([42102, 6])  (per graph)
  x NaN        : False  ✅
  x Inf        : False  ✅
  x range      : -2.6922 ~ 23.3808
  y NaN        : False  ✅
  y Inf        : False  ✅
  y range      : 0.001259 ~ 0.002377
  valid edges  : ✅

N28 C20 (SAME-DOMAIN TARGET)
  configs      : 271
  x shape      : torch.Size([41405, 6])  (per graph)
  x NaN        : False  ✅
  x Inf        : False  ✅
  x range      : -2.7371 ~ 23.4506
  y NaN        : False  ✅
  y Inf        : False  ✅
  y range      : 0.001315 ~ 0.006693
  valid edges  : ✅

N14     (CROSS-DOMAIN TARGET)
  configs      : 3441
  base x shape : torch.Size([34983, 2])
  config shape : (3441, 5)
  x NaN        : False  ✅
  x Inf        : False  ✅
  x range      : -1.7931 ~ 20.6836
  config NaN   : False  ✅
  config Inf   : False  ✅
  y NaN        : False  ✅
  y Inf        : False  ✅
  y range      : 0.002962 ~ 0.006666
  valid edges  : ✅

SCALER CHECK
  mean : [ 3.9944184 43.580032 ]


In [ ]:
import torch
import numpy as np

save_dir = '/content/drive/MyDrive/PEDA_Processed'

# C2 / C20 stay as before -- small enough to load as full Data lists.
data_c2_list  = torch.load(f'{save_dir}/data_c2_raw.pt',  weights_only=False)
data_c20_list = torch.load(f'{save_dir}/data_c20_raw.pt', weights_only=False)

# N14 uses the compact representation (fix from cell 29) -- labels
# live directly in n14_labels_raw.npy, no need to build 3441 Data
# objects just to compute mean/std.
n14_labels_raw = np.load(f'{save_dir}/n14_labels_raw.npy')

y_c2  = torch.cat([d.y for d in data_c2_list])
y_c20 = torch.cat([d.y for d in data_c20_list])
y_n14 = torch.tensor(n14_labels_raw)

c2_mean,  c2_std  = y_c2.mean(),  y_c2.std()
c20_mean, c20_std = y_c20.mean(), y_c20.std()
n14_mean, n14_std = y_n14.mean(), y_n14.std()

np.save(f'{save_dir}/label_c2_mean.npy',  c2_mean.numpy())
np.save(f'{save_dir}/label_c2_std.npy',   c2_std.numpy())
np.save(f'{save_dir}/label_c20_mean.npy', c20_mean.numpy())
np.save(f'{save_dir}/label_c20_std.npy',  c20_std.numpy())
np.save(f'{save_dir}/label_n14_mean.npy', n14_mean.numpy())
np.save(f'{save_dir}/label_n14_std.npy',  n14_std.numpy())

print("Per-domain label stats (for optional use at training time):")
print(f"  C2  : mean={c2_mean:.6f}  std={c2_std:.6f}")
print(f"  C20 : mean={c20_mean:.6f}  std={c20_std:.6f}")
print(f"  N14 : mean={n14_mean:.6f}  std={n14_std:.6f}")
print(f"\n\u2705 label_c2/c20/n14_mean.npy and _std.npy saved.")



Per-domain label stats (for optional use at training time):
  C2  : mean=0.001838  std=0.000381
  C20 : mean=0.001998  std=0.000759
  N14 : mean=0.004918  std=0.000874

✅ label_c2/c20/n14_mean.npy and _std.npy saved.


In [21]:
import os

save_dir = '/content/drive/MyDrive/PEDA_Processed'

# Delete leftovers from earlier (pre-fix / abandoned) attempts:
# old single-graph files, and the abandoned global label scaler.
for f in ['data_c2.pt', 'data_c20.pt', 'data_n14.pt',
          'data_c2_list.pt', 'data_c20_list.pt', 'data_n14_list.pt',
          'label_mean.npy', 'label_std.npy']:
    path = f'{save_dir}/{f}'
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")
    else:
        print(f"Not found (OK): {f}")

print(f"\n=== Final Drive contents ===")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(f'{save_dir}/{f}') / (1024 * 1024)
    print(f"  {f} ({size:.1f} MB)")


Not found (OK): data_c2.pt
Not found (OK): data_c20.pt
Not found (OK): data_n14.pt
Not found (OK): data_c2_list.pt
Not found (OK): data_c20_list.pt
Not found (OK): data_n14_list.pt
Not found (OK): label_mean.npy
Not found (OK): label_std.npy

=== Final Drive contents ===
  config_mean.npy (0.0 MB)
  config_std.npy (0.0 MB)
  data_c20_raw.pt (260.6 MB)
  data_c2_raw.pt (258.2 MB)
  label_c20_mean.npy (0.0 MB)
  label_c20_std.npy (0.0 MB)
  label_c2_mean.npy (0.0 MB)
  label_c2_std.npy (0.0 MB)
  label_n14_mean.npy (0.0 MB)
  label_n14_std.npy (0.0 MB)
  n14_base_x.pt (0.3 MB)
  n14_config_mean.npy (0.0 MB)
  n14_config_std.npy (0.0 MB)
  n14_configs_norm.npy (0.1 MB)
  n14_edge_index.pt (3.1 MB)
  n14_labels_raw.npy (0.0 MB)
  scaler_mean.npy (0.0 MB)
  scaler_std.npy (0.0 MB)


In [ ]:
# ============================================
# FINAL VERIFICATION CELL
# Run this to confirm all files are correct
# ============================================

import torch
import numpy as np
import os

save_dir = '/content/drive/MyDrive/PEDA_Processed'

print("=" * 50)
print("  FILE CHECK")
print("=" * 50)

required = [
    'data_c2_raw.pt', 'data_c20_raw.pt',
    'n14_base_x.pt', 'n14_edge_index.pt', 'n14_configs_norm.npy', 'n14_labels_raw.npy',
    'scaler_mean.npy', 'scaler_std.npy',
    'config_mean.npy', 'config_std.npy',
    'n14_config_mean.npy', 'n14_config_std.npy',
    'label_c2_mean.npy', 'label_c2_std.npy',
    'label_c20_mean.npy', 'label_c20_std.npy',
    'label_n14_mean.npy', 'label_n14_std.npy',
]

all_ok = True
for f in required:
    path = f'{save_dir}/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"  \u2705 {f:<30} ({size:.1f} MB)")
    else:
        print(f"  \u274c {f:<30} MISSING!")
        all_ok = False

print("\n" + "=" * 50)
print("  DATA CHECK")
print("=" * 50)

data_c2_list  = torch.load(f'{save_dir}/data_c2_raw.pt',  weights_only=False)
data_c20_list = torch.load(f'{save_dir}/data_c20_raw.pt', weights_only=False)
n14_base_x     = torch.load(f'{save_dir}/n14_base_x.pt')
n14_edge_index = torch.load(f'{save_dir}/n14_edge_index.pt')
n14_labels     = np.load(f'{save_dir}/n14_labels_raw.npy')

for name, dl in [("C2 ", data_c2_list), ("C20", data_c20_list)]:
    y_all = torch.cat([d.y for d in dl])
    x_all = torch.cat([d.x for d in dl], dim=0)
    x_ok  = not torch.isnan(x_all).any() and not torch.isinf(x_all).any()
    y_ok  = not torch.isnan(y_all).any() and not torch.isinf(y_all).any()
    ei_ok = all(d.edge_index.min().item() >= 0 and d.edge_index.max().item() < d.x.shape[0] for d in dl)
    print(f"  {name} | configs={len(dl):>5} | nodes/graph={dl[0].x.shape[0]:>6} | "
          f"x={'\u2705' if x_ok else '\u274c'} "
          f"y={'\u2705' if y_ok else '\u274c'} "
          f"edges={'\u2705' if ei_ok else '\u274c'} | "
          f"y range: {y_all.min():.4f}~{y_all.max():.4f}")

y_n14 = torch.tensor(n14_labels)
x_ok  = not torch.isnan(n14_base_x).any() and not torch.isinf(n14_base_x).any()
y_ok  = not torch.isnan(y_n14).any() and not torch.isinf(y_n14).any()
ei_ok = n14_edge_index.min().item() >= 0 and n14_edge_index.max().item() < n14_base_x.shape[0]
print(f"  N14 | configs={len(n14_labels):>5} | nodes/graph={n14_base_x.shape[0]:>6} | "
      f"x={'\u2705' if x_ok else '\u274c'} "
      f"y={'\u2705' if y_ok else '\u274c'} "
      f"edges={'\u2705' if ei_ok else '\u274c'} | "
      f"y range: {y_n14.min():.4f}~{y_n14.max():.4f}  (built on demand, not stored duplicated)")

print("\n" + "=" * 50)
print("  DOMAIN SHIFT CHECK")
print("=" * 50)
y_c2  = torch.cat([d.y for d in data_c2_list])
y_c20 = torch.cat([d.y for d in data_c20_list])
print(f"  C2  mean : {y_c2.mean():.6f}")
print(f"  C20 mean : {y_c20.mean():.6f}  (diff: {abs(y_c20.mean()-y_c2.mean()):.6f})")
print(f"  N14 mean : {y_n14.mean():.6f}  (diff: {abs(y_n14.mean()-y_c2.mean()):.6f})")

print("\n" + "=" * 50)
if all_ok:
    print("  \u2705 PREPROCESSING 100% COMPLETE!")
    print("  \u2705 All files correct \u2014 ready for GNN!")
else:
    print("  \u274c SOME FILES MISSING \u2014 check above!")
print("=" * 50)



#/////
print("\n" + "=" * 50)
print("  CROSS-DOMAIN COMPATIBILITY")
print("=" * 50)

n28_x_dim = data_c2_list[0].x.shape[1]
n14_x_dim = n14_base_x.shape[1]
n28_cfg_dim = data_c2_list[0].cfg.shape[1]

# load n14 cfg dim from the saved file for this cell's own scope
n14_cfg_dim = np.load(f'{save_dir}/n14_configs_norm.npy').shape[1]

print(f"  N28 x dimension   : {n28_x_dim}")
print(f"  N14 x dimension   : {n14_x_dim}")
print(f"  N28 cfg dimension : {n28_cfg_dim}")
print(f"  N14 cfg dimension : {n14_cfg_dim}")

assert n28_x_dim == n14_x_dim == 2, "Structural feature dims must match for frozen backbone transfer!"

print("  ✅ N28 and N14 structural feature dimensions match")
print("  ✅ Configuration features remain domain-specific")
print("  ✅ Frozen GNN backbone can accept both domains")


  FILE CHECK
  ✅ data_c2_raw.pt                 (270.8 MB)
  ✅ data_c20_raw.pt                (273.3 MB)
  ✅ n14_base_x.pt                  (0.3 MB)
  ✅ n14_edge_index.pt              (3.2 MB)
  ✅ n14_configs_norm.npy           (0.1 MB)
  ✅ n14_labels_raw.npy             (0.0 MB)
  ✅ scaler_mean.npy                (0.0 MB)
  ✅ scaler_std.npy                 (0.0 MB)
  ✅ config_mean.npy                (0.0 MB)
  ✅ config_std.npy                 (0.0 MB)
  ✅ n14_config_mean.npy            (0.0 MB)
  ✅ n14_config_std.npy             (0.0 MB)
  ✅ label_c2_mean.npy              (0.0 MB)
  ✅ label_c2_std.npy               (0.0 MB)
  ✅ label_c20_mean.npy             (0.0 MB)
  ✅ label_c20_std.npy              (0.0 MB)
  ✅ label_n14_mean.npy             (0.0 MB)
  ✅ label_n14_std.npy              (0.0 MB)

  DATA CHECK
  C2  | configs=  264 | nodes/graph= 42102 | x=✅ y=✅ edges=✅ | y range: 0.0013~0.0024
  C20 | configs=  271 | nodes/graph= 41405 | x=✅ y=✅ edges=✅ | y range: 0.0013~0.0067
  N14